# HINDSIGHT · Midterm: KMeans keyframe selection on SuperMemory-VQA

**Goal (unsupervised part of the project).** Long egocentric videos have thousands of frames, and captioning or VLM calls on every one are too expensive.
We cluster frames with **KMeans on CLIP embeddings** and keep one representative frame per cluster.
Then we check whether those few frames still cover the moments that answer **Object & Location Memory** questions, compared with **uniform** and **random** sampling at the same frame budget.

**Pipeline**
1. Load the QA labels and pick the sessions that contain the most object-question evidence.
2. Download those videos and sample frames at **1 fps, 224 px** *(preprocessing 1)*.
3. Embed frames with **CLIP ViT-B/32** *(preprocessing 2)*.
4. **PCA** to 64 dimensions *(preprocessing 3)*.
5. **KMeans** sweep over k: elbow and silhouette plots. Two variants: **A** uses appearance only; **B** uses appearance plus time.
6. Keyframes = the frame nearest each cluster centre. **Evidence coverage vs frame budget**: KMeans vs uniform vs random.
7. Cluster quality vs the dataset's room labels (NMI/ARI).
8. Visualizations for the report: t-SNE, timeline strip, cluster gallery.
9. Save outputs to Drive; optionally push compressed frames and embeddings to a **private** Hugging Face repo.

### Before you run
- **Runtime → Change runtime type → T4 GPU** (an L4 is faster if you have it; an A100 isn't needed).
- **Hugging Face token:** click the 🔑 **Secrets** icon in the left sidebar, add a secret named `HF_TOKEN`, and turn on *Notebook access*. Use a *read* token, or a *write* token if you want step 9 to push to HF.
- Disk: each raw video is 1.5–3 GB. They are deleted after frame extraction (`DELETE_RAW = True`).

In [ ]:
# ===================== CONFIG =====================
N_SESSIONS     = 6          # how many sessions to process (picked by most object-question evidence, max 1 per person)
ONE_PER_PERSON = True       # spread sessions across participants (useful later for a person-wise train/test split)
FPS            = 1          # frames per second to sample
IMG            = 224        # frame size (px)
PCA_DIM        = 64
BUDGETS        = [0.005, 0.01, 0.02, 0.05, 0.10]   # fraction of frames kept
CLUSTERS_PER_MIN = [0.5, 1, 2, 4, 8]                # k sweep, scaled by session length
TIME_WEIGHT    = 1.0        # variant B: weight of the time feature
TOL_S          = 2.0        # a keyframe within +/- TOL_S seconds of an evidence span counts as a hit
RANDOM_SEEDS   = 5
DELETE_RAW     = True       # delete the raw mp4 after extracting frames
SAVE_TO_DRIVE  = True       # copy frames/embeddings/results/figures to Google Drive
DRIVE_DIR      = "/content/drive/MyDrive/hindsight_midterm"
PUSH_TO_HF     = False      # step 9: push compressed frames + embeddings to a PRIVATE HF dataset repo
HF_PUSH_REPO   = "YOUR_HF_USERNAME/hindsight-supermemory-1fps"   # change before enabling PUSH_TO_HF

REPO_ID  = "OSU-AIoT-MLSys-Lab/SuperMemory-VQA"
WORK     = "/content/work"
# ==================================================
import os
for d in ["raw", "frames", "emb", "fig", "results"]:
    os.makedirs(f"{WORK}/{d}", exist_ok=True)

In [ ]:
!pip -q install open_clip_torch
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU attached: Runtime > Change runtime type > T4 GPU"
!ffmpeg -version | head -1

In [ ]:
# ---- Hugging Face login (token stored in Colab Secrets as HF_TOKEN) ----
import os
from huggingface_hub import login
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:                       # not on Colab: fall back to env var or prompt
    HF_TOKEN = os.environ.get("HF_TOKEN") or __import__("getpass").getpass("HF token: ")
login(token=HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN
print("Logged in to Hugging Face.")

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)

## Step 1: Load QA labels and pick sessions

Timestamps: evidence `time_span.start_time / end_time` are **seconds from the start of that video**.
Answer evidence is usually in an *earlier* session than the question (cross-session memory).
When the evidence is in the *same* session as the question, only frames before the question time may be used.

In [ ]:
import json, re, numpy as np, pandas as pd
from huggingface_hub import hf_hub_download, list_repo_files

qa_path = hf_hub_download(REPO_ID, "data/json/all_qa.json", repo_type="dataset")
QA = json.load(open(qa_path))
obj = [r for r in QA if r["metadata"]["skill"] == "object_location_memory"]
print(f"{len(QA)} questions total, {len(obj)} object & location, {sum(r['is_answerable'] for r in obj)} answerable")

def to_sec(x):
    # times are usually seconds (float) but a few records use "MM:SS" or "HH:MM:SS" strings
    if isinstance(x, (int, float)):
        return float(x)
    sec = 0.0
    for part in str(x).strip().split(":"):
        sec = sec * 60 + float(part)
    return sec

def spans_of(ev):
    # question_evidence uses "time_spans" (list) in most records, "time_span" (dict) in a few
    sp = ev.get("time_spans") or ([ev["time_span"]] if ev.get("time_span") else [])
    return [(to_sec(s["start_time"]), to_sec(s["end_time"])) for s in sp]

# one row per answer-evidence span
rows = []
for r in obj:
    if not r["is_answerable"]:
        continue
    prim = r["metadata"]["primary_video_id"]
    q_spans = spans_of(r["question_evidence"])
    q_time = min(s for s, _ in q_spans) if q_spans else np.inf
    for e in r["answer_evidence"]["evidence_list"]:
        (s0, e0), = spans_of(e)
        rows.append(dict(qid=r["question_id"], person=r["subject"], video_id=e["video_id"],
                         s=s0, e=e0,
                         room=e.get("room", ""), q_cut=(q_time if e["video_id"] == prim else np.inf),
                         question=r["question"]))
EV = pd.DataFrame(rows)

# map video_id -> mp4 path in the repo
mp4s = [f for f in list_repo_files(REPO_ID, repo_type="dataset") if f.endswith(".mp4")]
VIDEO_PATH = {os.path.basename(f)[:-4]: f for f in mp4s}
EV = EV[EV.video_id.isin(VIDEO_PATH)]

counts = EV.groupby(["video_id", "person"]).size().reset_index(name="n_spans").sort_values("n_spans", ascending=False)
if ONE_PER_PERSON:
    counts = counts.drop_duplicates("person")
SESSIONS = counts.head(N_SESSIONS).video_id.tolist()
display(counts.head(N_SESSIONS))
EV_SEL = EV[EV.video_id.isin(SESSIONS)].reset_index(drop=True)
print(f"{len(EV_SEL)} evidence spans across {len(SESSIONS)} sessions")

## Step 2: Download videos and sample frames (1 fps, 224 px)
Each frame `i` (0-indexed) is at about `t = i / FPS` seconds. Roughly 3–8 minutes per session on Colab.

In [ ]:
import glob, shutil, subprocess, time
from tqdm.auto import tqdm

def extract_frames(video_id):
    out = f"{WORK}/frames/{video_id}"
    if os.path.isdir(out) and len(os.listdir(out)) > 0:
        return len(os.listdir(out))
    t0 = time.time()
    mp4 = hf_hub_download(REPO_ID, VIDEO_PATH[video_id], repo_type="dataset", local_dir=f"{WORK}/raw")
    os.makedirs(out, exist_ok=True)
    cmd = ["ffmpeg", "-loglevel", "error", "-threads", "0", "-i", mp4,
           "-vf", f"fps={FPS},scale={IMG}:{IMG}", "-q:v", "3", f"{out}/%06d.jpg"]
    subprocess.run(cmd, check=True)
    if DELETE_RAW:
        os.remove(mp4)
    n = len(os.listdir(out))
    print(f"{video_id}: {n} frames ({n/60/FPS:.1f} min) in {time.time()-t0:.0f}s")
    return n

N_FRAMES = {v: extract_frames(v) for v in SESSIONS}

## Step 3: CLIP embeddings

In [ ]:
import torch, open_clip
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, _, clip_prep = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
clip_model = clip_model.eval().to(device)

class Frames(torch.utils.data.Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return clip_prep(Image.open(self.paths[i]).convert("RGB"))

@torch.no_grad()
def embed_session(video_id, bs=256):
    f = f"{WORK}/emb/{video_id}.npy"
    if os.path.exists(f):
        return np.load(f)
    paths = sorted(glob.glob(f"{WORK}/frames/{video_id}/*.jpg"))
    dl = torch.utils.data.DataLoader(Frames(paths), batch_size=bs, num_workers=2)
    out = []
    for x in tqdm(dl, desc=video_id[:28]):
        with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
            e = clip_model.encode_image(x.to(device))
        out.append(torch.nn.functional.normalize(e.float(), dim=-1).cpu().numpy())
    E = np.concatenate(out).astype(np.float32)
    np.save(f, E)
    return E

EMB = {v: embed_session(v) for v in SESSIONS}
TIMES = {v: np.arange(len(EMB[v])) / FPS for v in SESSIONS}
{v: EMB[v].shape for v in SESSIONS}

## Step 4: PCA (fit per session)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

Z, EVR = {}, {}
for v in SESSIONS:
    p = PCA(n_components=PCA_DIM, random_state=0).fit(EMB[v])
    Z[v] = p.transform(EMB[v]).astype(np.float32)
    EVR[v] = np.cumsum(p.explained_variance_ratio_)

plt.figure(figsize=(6, 4))
for v in SESSIONS:
    plt.plot(range(1, PCA_DIM + 1), EVR[v], label=v.split("_glasses")[0])
plt.xlabel("PCA components"); plt.ylabel("Cumulative explained variance")
plt.title("PCA on CLIP frame embeddings"); plt.legend(fontsize=7); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{WORK}/fig/pca_variance.png", dpi=150); plt.show()
print({v.split('_glasses')[0]: round(float(EVR[v][-1]), 3) for v in SESSIONS})

## Step 5: KMeans and choosing k
- **Variant A:** appearance only (PCA features).
- **Variant B:** appearance plus a scaled time column, so clusters also stay together in time and behave more like scenes.

k is expressed as **clusters per minute** so it scales with session length.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

def feats(v, w_time):
    X = Z[v]
    if w_time == 0:
        return X
    t = TIMES[v]; tt = (t - t.min()) / (t.max() - t.min() + 1e-9)
    scale = np.sqrt(X.var(0).sum())          # put time on the same scale as the appearance features
    return np.hstack([X, (w_time * scale * tt)[:, None]]).astype(np.float32)

sweep = []
for v in tqdm(SESSIONS, desc="k sweep"):
    minutes = len(Z[v]) / FPS / 60
    for variant, w in [("A: appearance", 0.0), ("B: appearance+time", TIME_WEIGHT)]:
        X = feats(v, w)
        for cpm in CLUSTERS_PER_MIN:
            k = int(np.clip(round(cpm * minutes), 2, len(X) - 1))
            km = KMeans(n_clusters=k, n_init=3, random_state=0).fit(X)
            sil = silhouette_score(X, km.labels_, sample_size=min(3000, len(X)), random_state=0)
            sweep.append(dict(video_id=v, variant=variant, cpm=cpm, k=k,
                              inertia_per_frame=km.inertia_ / len(X), silhouette=sil))
SWEEP = pd.DataFrame(sweep)
SWEEP.to_csv(f"{WORK}/results/k_sweep.csv", index=False)

agg = SWEEP.groupby(["variant", "cpm"])[["inertia_per_frame", "silhouette"]].mean().reset_index()
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for variant, g in agg.groupby("variant"):
    ax[0].plot(g.cpm, g.inertia_per_frame, "o-", label=variant)
    ax[1].plot(g.cpm, g.silhouette, "o-", label=variant)
ax[0].set(xscale="log", xlabel="clusters per minute", ylabel="inertia / frame", title="Elbow")
ax[1].set(xscale="log", xlabel="clusters per minute", ylabel="silhouette", title="Silhouette")
for a in ax: a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.savefig(f"{WORK}/fig/elbow_silhouette.png", dpi=150); plt.show()
agg

## Step 6: Keyframes and evidence coverage vs frame budget

For each budget (fraction of frames kept), every method keeps the **same number of frames**:
- **Uniform:** evenly spaced frames.
- **Random:** averaged over several seeds.
- **KMeans A / B:** k = budget × frames; keep the frame nearest each centroid.

**Evidence coverage** is the fraction of answer-evidence spans with at least one kept frame inside `[start − TOL_S, end + TOL_S]` that is also before the question time when the evidence is in the question's own session.

In [ ]:
def pick_uniform(n, b):
    return np.unique(np.linspace(0, n - 1, b).round().astype(int))

def pick_random(n, b, seed):
    return np.sort(np.random.default_rng(seed).choice(n, size=b, replace=False))

def pick_kmeans(v, b, w_time):
    X = feats(v, w_time)
    km = KMeans(n_clusters=b, n_init=3, random_state=0).fit(X)
    keep = []
    for c in range(b):
        idx = np.where(km.labels_ == c)[0]
        if len(idx):
            keep.append(idx[np.linalg.norm(X[idx] - km.cluster_centers_[c], axis=1).argmin()])
    return np.sort(np.array(keep))

def span_hits(kept_idx, v):
    # returns a boolean array: one entry per evidence span in session v
    kt = TIMES[v][kept_idx]
    ev = EV_SEL[EV_SEL.video_id == v]
    hits = []
    for s, e, qc in zip(ev.s, ev.e, ev.q_cut):
        ok = (kt >= s - TOL_S) & (kt <= e + TOL_S) & (kt <= qc)
        hits.append(bool(ok.any()))
    return np.array(hits)

res = []
for v in tqdm(SESSIONS, desc="coverage"):
    n = len(Z[v])
    for frac in BUDGETS:
        b = max(2, int(round(frac * n)))
        res.append(dict(video_id=v, budget=frac, b=b, method="Uniform", coverage=span_hits(pick_uniform(n, b), v).mean()))
        res.append(dict(video_id=v, budget=frac, b=b, method="Random",
                        coverage=np.mean([span_hits(pick_random(n, b, s), v).mean() for s in range(RANDOM_SEEDS)])))
        res.append(dict(video_id=v, budget=frac, b=b, method="KMeans A (appearance)", coverage=span_hits(pick_kmeans(v, b, 0.0), v).mean()))
        res.append(dict(video_id=v, budget=frac, b=b, method="KMeans B (+time)", coverage=span_hits(pick_kmeans(v, b, TIME_WEIGHT), v).mean()))
COV = pd.DataFrame(res)
COV.to_csv(f"{WORK}/results/coverage.csv", index=False)

# weight sessions by number of evidence spans (= micro-average over spans)
w = EV_SEL.groupby("video_id").size()
COV["w"] = COV.video_id.map(w)
summary = (COV.groupby(["method", "budget"])
              .apply(lambda g: np.average(g.coverage, weights=g.w), include_groups=False)
              .unstack("budget"))
display((summary * 100).round(1))

plt.figure(figsize=(6.5, 4.5))
for m in ["Random", "Uniform", "KMeans A (appearance)", "KMeans B (+time)"]:
    plt.plot(np.array(BUDGETS) * 100, summary.loc[m].values * 100, "o-", label=m)
plt.xscale("log"); plt.xticks(np.array(BUDGETS) * 100, [f"{b*100:g}%" for b in BUDGETS]); plt.xlabel("Frames kept (% of 1-fps frames)"); plt.ylabel("Evidence spans covered (%)")
plt.title("Object & Location Memory: evidence coverage vs frame budget"); plt.grid(alpha=.3); plt.legend()
plt.tight_layout(); plt.savefig(f"{WORK}/fig/coverage_vs_budget.png", dpi=150); plt.show()

In [ ]:
# frame reduction at each budget (for the report)
red = COV.drop_duplicates(["video_id", "budget"])[["video_id", "budget", "b"]].copy()
red["frames"] = red.video_id.map(lambda v: len(Z[v]))
red["reduction_x"] = (red.frames / red.b).round(1)
display(red.groupby("budget")[["frames", "b", "reduction_x"]].mean().round(1))

## Step 7: Do clusters match rooms? (NMI / ARI)
Evidence spans carry a free-text `room` label (for example "Kitchen island"). We map it to a coarse room and compare it with cluster IDs for frames inside evidence spans.
This is approximate because the room labels are free text and only cover evidence frames.

In [ ]:
from sklearn.metrics import normalized_mutual_info_score as NMI, adjusted_rand_score as ARI

ROOM_MAP = [("kitchen", "kitchen"), ("living", "living"), ("couch", "living"), ("sofa", "living"),
            ("dining", "dining"), ("bath", "bathroom"), ("bed", "bedroom"), ("hall", "hallway"),
            ("entr", "hallway"), ("laundry", "laundry"), ("office", "office"), ("desk", "office"),
            ("garage", "garage"), ("car", "car"), ("store", "store"), ("shop", "store"),
            ("outdoor", "outdoor"), ("outside", "outdoor"), ("patio", "outdoor"), ("yard", "outdoor"),
            ("porch", "outdoor"), ("street", "outdoor"), ("park", "outdoor")]
def coarse_room(s):
    s = (s or "").lower()
    for key, lab in ROOM_MAP:
        if key in s:
            return lab
    return None

CPM_FOR_NMI = 1     # clusters per minute used for the NMI check
nmi_rows = []
for v in SESSIONS:
    ev = EV_SEL[EV_SEL.video_id == v]
    t = TIMES[v]; lab = np.array([None] * len(t), dtype=object)
    for s, e, room in zip(ev.s, ev.e, ev.room):
        lab[(t >= s) & (t <= e)] = coarse_room(room)
    m = np.array([l is not None for l in lab])
    if m.sum() < 20 or len(set(lab[m])) < 2:
        continue
    k = max(2, int(round(CPM_FOR_NMI * len(t) / FPS / 60)))
    for variant, wt in [("A", 0.0), ("B", TIME_WEIGHT)]:
        km = KMeans(n_clusters=k, n_init=3, random_state=0).fit(feats(v, wt))
        rnd = np.random.default_rng(0).permutation(km.labels_)
        nmi_rows.append(dict(video_id=v, variant=variant, frames=int(m.sum()), rooms=len(set(lab[m])),
                             NMI=NMI(lab[m], km.labels_[m]), ARI=ARI(lab[m], km.labels_[m]),
                             NMI_shuffled=NMI(lab[m], rnd[m])))
NMI_DF = pd.DataFrame(nmi_rows)
NMI_DF.to_csv(f"{WORK}/results/nmi_rooms.csv", index=False)
display(NMI_DF.round(3) if len(NMI_DF) else "Not enough room-labelled frames")
if len(NMI_DF): display(NMI_DF.groupby("variant")[["NMI", "ARI", "NMI_shuffled"]].mean().round(3))

## Step 8: Visualizations for the report

In [ ]:
from sklearn.manifold import TSNE

v = SESSIONS[0]                          # session with the most evidence
k = max(2, int(round(1 * len(Z[v]) / FPS / 60)))   # 1 cluster per minute
km_v = KMeans(n_clusters=k, n_init=3, random_state=0).fit(feats(v, TIME_WEIGHT))
labels_v = km_v.labels_

idx = np.random.default_rng(0).choice(len(Z[v]), size=min(2000, len(Z[v])), replace=False)
T2 = TSNE(n_components=2, init="pca", perplexity=30, random_state=0).fit_transform(Z[v][idx])

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].scatter(T2[:, 0], T2[:, 1], c=labels_v[idx] % 20, cmap="tab20", s=5)
ax[0].set_title(f"t-SNE of frames, coloured by KMeans cluster (k={k})")
sc = ax[1].scatter(T2[:, 0], T2[:, 1], c=TIMES[v][idx] / 60, cmap="viridis", s=5)
ax[1].set_title("Same points coloured by time (min)"); plt.colorbar(sc, ax=ax[1])
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.suptitle(v); plt.tight_layout(); plt.savefig(f"{WORK}/fig/tsne_clusters.png", dpi=150); plt.show()

# timeline strip: cluster ID over time, with evidence spans marked
fig, ax = plt.subplots(figsize=(12, 1.8))
ax.imshow((labels_v % 20)[None, :], aspect="auto", cmap="tab20",
          extent=[0, TIMES[v][-1] / 60, 0, 1], interpolation="nearest")
for s, e in EV_SEL[EV_SEL.video_id == v][["s", "e"]].values:
    ax.axvspan(s / 60, e / 60, ymin=1.02, ymax=1.25, color="k", clip_on=False)
ax.set_yticks([]); ax.set_xlabel("minutes"); ax.set_title("KMeans cluster over time (black bars = answer-evidence spans)", pad=24)
plt.tight_layout(); plt.savefig(f"{WORK}/fig/timeline_clusters.png", dpi=150); plt.show()

In [ ]:
# cluster gallery: the 6 largest clusters, 6 frames each nearest the centroid
X = feats(v, TIME_WEIGHT)
paths = sorted(glob.glob(f"{WORK}/frames/{v}/*.jpg"))
top = np.argsort(np.bincount(labels_v))[::-1][:6]
fig, axes = plt.subplots(len(top), 6, figsize=(12, 2.1 * len(top)))
for r, c in enumerate(top):
    members = np.where(labels_v == c)[0]
    near = members[np.argsort(np.linalg.norm(X[members] - km_v.cluster_centers_[c], axis=1))][:6]
    for j in range(6):
        a = axes[r, j]; a.axis("off")
        if j < len(near):
            a.imshow(Image.open(paths[near[j]]))
            a.set_title(f"c{c} · {TIMES[v][near[j]]/60:.1f} min", fontsize=7)
plt.suptitle("Largest KMeans clusters (frames nearest the centroid)")
plt.tight_layout(); plt.savefig(f"{WORK}/fig/cluster_gallery.png", dpi=130); plt.show()

## Step 9: Save outputs (Drive) and optionally push to a private HF repo
Frames and embeddings are pushed only if `PUSH_TO_HF = True`. Keep the repo **private**: the dataset is CC BY-NC-SA 4.0, so credit SuperMemory-VQA and keep the same license.

In [ ]:
if SAVE_TO_DRIVE:
    for sub in ["emb", "fig", "results"]:
        shutil.copytree(f"{WORK}/{sub}", f"{DRIVE_DIR}/{sub}", dirs_exist_ok=True)
    os.makedirs(f"{DRIVE_DIR}/frames", exist_ok=True)
    for v in SESSIONS:                        # frames as one tar per session (fast to copy)
        tar = f"{DRIVE_DIR}/frames/{v}.tar"
        if not os.path.exists(tar):
            shutil.make_archive(tar[:-4], "tar", f"{WORK}/frames", v)
    print("Saved to", DRIVE_DIR)

if PUSH_TO_HF:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_PUSH_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(repo_id=HF_PUSH_REPO, repo_type="dataset", folder_path=f"{WORK}/emb", path_in_repo="emb")
    api.upload_folder(repo_id=HF_PUSH_REPO, repo_type="dataset", folder_path=f"{WORK}/frames", path_in_repo="frames_1fps_224")
    print("Pushed to https://huggingface.co/datasets/" + HF_PUSH_REPO)

## What goes in the midterm report
- **Methods:** 1-fps sampling at 224 px → CLIP ViT-B/32 → PCA (64 dimensions, report explained variance) → KMeans per session (A: appearance; B: plus time) → centroid keyframes. Explain *why*: captioning or VLM calls on every frame are too expensive.
- **Results:** `coverage_vs_budget.png` (headline), the coverage table, the frame-reduction table, `elbow_silhouette.png`, NMI vs rooms.
- **Visuals:** t-SNE, timeline strip, cluster gallery.
- **Discussion:** where KMeans beats uniform (long static stretches collapse into one cluster); where it fails (small changes, such as keys appearing on a table, share a cluster with the scene, so the centroid frame can miss the moment).
- **Next steps:** caption objects in the kept frames → supervised scorers (cosine vs LogReg / RF / MLP) with evidence-span labels, split by participant → answerability classifier → VLM answers vs baselines.
- **Note:** the dataset card asks that systems trained on SuperMemory-VQA labels report this separately, so say the scorers are trained on part of the participants and tested on held-out participants.